# Pollution Model Vetting & Diagnostics

This notebook prepares and evaluates multiple pollution prediction models
to support decision-making in the Toxic Tides and EcoPulse modules.

The objectives are:
- Compare multiple scientifically defensible model configurations
- Preserve transparency around censored laboratory data
- Identify anomalous pollution observations
- Save validated models for downstream application use

This notebook does **not** optimize for maximum predictive performance,
but for interpretability, robustness, and regulatory credibility.

In [81]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [107]:
# Load Raw Datasets
pollution = pd.read_csv("../data/processed/squid_pollution.csv")
#print(pollution.head())
catch = pd.read_csv("../data/processed/squid_catch_env.csv")
#print(catch.head())
industrial_raw = pd.read_csv("../data/raw/industrial_output.csv")
print(industrial_raw.head())
agricultural_raw = pd.read_csv("../data/raw/agricultural_output.csv")
#print(agricultural_raw.head())

      Month    Country  Year  PercentChange
0   October  Argentina  2018           -8.4
1  November  Argentina  2018          -14.0
2  December  Argentina  2018          -14.8
3   October     Brazil  2018            0.7
4  November     Brazil  2018           -1.2


## Human Pressure Indices

Industrial output is reported monthly (% change).
Agricultural output is reported quarterly (GDP-based).

To combine them:
- Convert both to monthly timestamps
- Forward-fill agricultural values
- Normalize both to unit scale
- Keep them as **separate pressures**

In [83]:
# Map month names to numbers
month_map = {
    "January": 1, "February": 2, "March": 3,
    "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9,
    "October": 10, "November": 11, "December": 12
}

industrial = industrial_raw.copy()

industrial["Month_num"] = industrial["Month"].map(month_map)

industrial["Date"] = pd.to_datetime(
    industrial["Year"].astype(str) + "-" +
    industrial["Month_num"].astype(str) + "-01"
)

# Index relative to country mean (defensible normalization)
industrial["Industrial_Index"] = (
    industrial.groupby("Country")["PercentChange"]
    .transform(lambda x: 100 + (x - x.mean()))
)

industrial = industrial[
    ["Country", "Date", "Year", "Month_num", "Industrial_Index"]
].rename(columns={"Month_num": "Month"})

agricultural = agricultural_raw.copy()

# Extract year and quarter label
agricultural["Year"] = agricultural["Quarter"].str[:4].astype(int)
agricultural["Quarter_Label"] = agricultural["Quarter"].str[5:]

# Map quarter to representative mid-quarter month
quarter_to_month = {
    "Jan-Mar": 1,     # February
    "Apr-Jun": 4,     # May
    "Jul-Sept": 7,    # August
    "Oct-Dec": 10     # November
}

agricultural["Month"] = agricultural["Quarter_Label"].map(quarter_to_month)

# Create Date column (THIS is what resampling needs)
agricultural["Date"] = pd.to_datetime(
    agricultural["Year"].astype(str) + "-" +
    agricultural["Month"].astype(str) + "-01"
)

print(industrial.head())


     Country       Date  Year  Month  Industrial_Index
0  Argentina 2018-10-01  2018     10         91.379487
1  Argentina 2018-11-01  2018     11         85.779487
2  Argentina 2018-12-01  2018     12         84.979487
3     Brazil 2018-10-01  2018     10        100.854359
4     Brazil 2018-11-01  2018     11         98.954359


In [84]:
#Normalizing GDP
agricultural["Agricultural_Index"] = (
    agricultural.groupby("Country")["GDP"]
    .transform(lambda x: x / x.mean() * 100)
)

In [85]:
agricultural = agricultural.copy()

agricultural["Date"] = (
    pd.to_datetime(agricultural["Date"])
    .dt.to_period("M")
    .dt.to_timestamp()
)

agricultural_monthly = (
    agricultural
    .set_index("Date")
    .groupby("Country", group_keys=False)
    .resample("MS")
    .ffill()
    .reset_index()
)

agricultural_monthly["Year"] = agricultural_monthly["Date"].dt.year
agricultural_monthly["Month"] = agricultural_monthly["Date"].dt.month

agricultural_monthly = agricultural_monthly[
    ["Country", "Date", "Year", "Month", "Agricultural_Index"]
]


print(agricultural_monthly.head())

     Country       Date  Year  Month  Agricultural_Index
0  Argentina 2018-10-01  2018     10           70.802292
1  Argentina 2018-11-01  2018     11           70.802292
2  Argentina 2018-12-01  2018     12           70.802292
3  Argentina 2019-01-01  2019      1           81.432665
4  Argentina 2019-02-01  2019      2           81.432665


In [86]:
pressure = industrial.merge(
    agricultural_monthly,
    on=["Country", "Year", "Month", "Date"],
    how="left"
)

pressure = pressure.sort_values(["Country", "Date"])

print(pressure.head())

      Country       Date  Year  Month  Industrial_Index  Agricultural_Index
0   Argentina 2018-10-01  2018     10         91.379487           70.802292
1   Argentina 2018-11-01  2018     11         85.779487           70.802292
2   Argentina 2018-12-01  2018     12         84.979487           70.802292
9   Argentina 2019-01-01  2019      1         88.579487           81.432665
10  Argentina 2019-02-01  2019      2         91.379487           81.432665


## Final Pressure Dataset

Each row represents:
- One country
- One month

Columns:
- Industrial_Index (monthly, normalized)
- Agricultural_Index (quarterly → monthly, normalized)

This table is now ready to be:
- Lagged
- Distance-weighted
- Merged with squid observations


In [87]:
pressure.to_csv("../data/processed/pressure_indices_monthly.csv", index=False)

## Environmental & Fishing Conditions

Environmental variables and squid catch are merged by year and month.
Lagged variables represent recent historical exposure.

In [108]:
# Prepare Catch Data
catch = catch.sort_values(["Year", "Month"])
print(catch.head())

for col in ["WaterTemp", "SSH", "Chlor_a_mg_m3", "SqCatch_Kg"]:
    catch[f"{col}_lag1"] = catch[col].shift(1)



   Year  Month  WaterTemp       Depth     SSH  Chlor_a_mg_m3    SqCatch_Kg  \
0  2018      2  13.309679  113.710177  0.0771       1.561093  1.149292e+07   
1  2018      3  12.477122  130.445444  0.0699       0.705288  3.380751e+07   
2  2018      4  10.536637  139.568540  0.0690       0.496143  1.068107e+07   
3  2018      5   9.499668  138.930233  0.0675       0.256468  1.175480e+06   
4  2019      1  15.333333  193.333333  0.0786       1.207356  2.974881e+02   

   WaterTemp_lag1  SSH_lag1  Chlor_a_mg_m3_lag1  SqCatch_Kg_lag1  
0       13.963235    0.0771            1.227153     3.715857e+04  
1       13.309679    0.0771            1.561093     1.149292e+07  
2       12.477122    0.0699            0.705288     3.380751e+07  
3       10.536637    0.0690            0.496143     1.068107e+07  
4        9.499668    0.0675            0.256468     1.175480e+06  


## Pollution Dataset Structure

Each row represents:
- One squid
- One tissue
- One pollutant

Censoring status (BLOD / BLOQ) is retained as a binary indicator.

In [121]:
pollution["is_censored"] = pollution["status"].isin(["BLOD", "BLOQ"]).astype(int)

pollution["Year"] = pollution["Year"].astype(int)
pollution["Month"] = pollution["Month_of_Capture"].astype(int)
print(pollution.head())

             ID  Year  Gender Latitude Longitude  Month_of_Capture  \
0  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
1  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
2  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
3  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
4  49_14_muscle  2021       1  46°55'S   60°51'W                 5   

   Mantle_length_mm  Wet_Weight_g  Maturity_level                       size  \
0             181.0         119.6               2  small (181 mm - 237.3 mm)   
1             181.0         119.6               2  small (181 mm - 237.3 mm)   
2             181.0         119.6               2  small (181 mm - 237.3 mm)   
3             181.0         119.6               2  small (181 mm - 237.3 mm)   
4             181.0         119.6               2  small (181 mm - 237.3 mm)   

   ... pollutant concentration    status outlier  dta_km  dtfl_km   dtu_km  \
0  ...   Metal_A    

## Final Modeling Dataset

We now merge:
- Pollution observations
- Environmental & catch conditions
- Human pressure indicators

This produces the **single authoritative dataset**
used for feature selection and modeling.

In [112]:
# ----------------------------
# Pivot Pressure by Country
# ----------------------------

# Pivot industrial index
ind_pivot = (
    pressure
    .pivot(index=["Year", "Month"],
           columns="Country",
           values="Industrial_Index")
    .add_prefix("Industrial_Index_")
    .reset_index()
)

# Pivot agricultural index
agr_pivot = (
    pressure
    .pivot(index=["Year", "Month"],
           columns="Country",
           values="Agricultural_Index")
    .add_prefix("Agricultural_Index_")
    .reset_index()
)

# Merge industrial + agricultural
pressure_wide = ind_pivot.merge(
    agr_pivot,
    on=["Year", "Month"],
    how="left"
)
print(pressure_wide.head())

Country  Year  Month  Industrial_Index_Argentina  Industrial_Index_Brazil  \
0        2018     10                   91.379487               100.854359   
1        2018     11                   85.779487                98.954359   
2        2018     12                   84.979487                96.354359   
3        2019      1                   88.579487                98.254359   
4        2019      2                   91.379487               102.454359   

Country  Industrial_Index_Uruguay  Agricultural_Index_Argentina  \
0                      117.169744                     70.802292   
1                       99.369744                     70.802292   
2                       88.869744                     70.802292   
3                       98.969744                     81.432665   
4                       95.169744                     81.432665   

Country  Agricultural_Index_Brazil  Agricultural_Index_Uruguay  
0                        66.852562                  130.853927  
1   

In [120]:
# ----------------------------
# Build Final Dataset
# ----------------------------

final_df = (
    pollution
    .merge(
        catch,
        left_on=["Year", "Month"],
        right_on=["Year", "Month"],
        how="left"
    )
    .merge(
        pressure_wide,
        on=["Year", "Month"],
        how="left"
    )
)

# Keep only rows with observed concentrations
final_df = final_df.dropna(subset=["concentration"])

# ----------------------------
# Physics-Informed Distance Weighting
# ----------------------------

epsilon = 1  # prevents divide-by-zero issues

# Industrial inverse-square pressures
final_df["Ind_ARG"] = final_df["Industrial_Index_Argentina"] / ((final_df["dta_km"] + epsilon) ** 2)
final_df["Ind_BRA"] = final_df["Industrial_Index_Brazil"] / ((final_df["dtb_km"] + epsilon) ** 2)
final_df["Ind_URU"] = final_df["Industrial_Index_Uruguay"] / ((final_df["dtu_km"] + epsilon) ** 2)

final_df["Industrial_Pressure"] = (
    final_df["Ind_ARG"] +
    final_df["Ind_BRA"] +
    final_df["Ind_URU"]
)

# Agricultural inverse-square pressures
final_df["Agr_ARG"] = final_df["Agricultural_Index_Argentina"] / ((final_df["dta_km"] + epsilon) ** 2)
final_df["Agr_BRA"] = final_df["Agricultural_Index_Brazil"] / ((final_df["dtb_km"] + epsilon) ** 2)
final_df["Agr_URU"] = final_df["Agricultural_Index_Uruguay"] / ((final_df["dtu_km"] + epsilon) ** 2)

final_df["Agricultural_Pressure"] = (
    final_df["Agr_ARG"] +
    final_df["Agr_BRA"] +
    final_df["Agr_URU"]
)

# ----------------------------
# Log-transform target
# ----------------------------

final_df["log_concentration"] = np.log1p(final_df["concentration"])

final_df.to_csv("../data/processed/final_modeling_dataset.csv", index=False)
print(final_df.head())



             ID  Year  Gender Latitude Longitude  Month_of_Capture  \
0  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
1  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
2  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
3  49_14_muscle  2021       1  46°55'S   60°51'W                 5   
4  49_14_muscle  2021       1  46°55'S   60°51'W                 5   

   Mantle_length_mm  Wet_Weight_g  Maturity_level                       size  \
0             181.0         119.6               2  small (181 mm - 237.3 mm)   
1             181.0         119.6               2  small (181 mm - 237.3 mm)   
2             181.0         119.6               2  small (181 mm - 237.3 mm)   
3             181.0         119.6               2  small (181 mm - 237.3 mm)   
4             181.0         119.6               2  small (181 mm - 237.3 mm)   

   ... Agricultural_Index_Uruguay  Ind_ARG  Ind_BRA   Ind_URU  \
0  ...                 105.505261

## Model Feature Configurations

Four vetted model configurations are evaluated:

1. **Environment Only**
   Captures physical and oceanographic drivers.

2. **Environment + Catch**
   Adds fishing pressure as a biological stressor.

3. **Full Pressures**
   Incorporates industrial and agricultural pressures.

4. **Full Pressures + Censoring (Sensitivity Model)**
   Includes laboratory censoring information for diagnostic purposes.



In [114]:
FEATURE_SETS = {
    "environment_only": [
        "WaterTemp", "Depth", "SSH", "Chlor_a_mg_m3",
        "WaterTemp_lag1", "SSH_lag1", "Chlor_a_mg_m3_lag1"
    ],

    "env_plus_catch": [
        "WaterTemp", "Depth", "SSH", "Chlor_a_mg_m3",
        "WaterTemp_lag1", "SSH_lag1", "Chlor_a_mg_m3_lag1",
        "SqCatch_Kg", "SqCatch_Kg_lag1"
    ],

    "full_pressures": [
        "WaterTemp", "Depth", "SSH", "Chlor_a_mg_m3",
        "WaterTemp_lag1", "SSH_lag1", "Chlor_a_mg_m3_lag1",
        "SqCatch_Kg", "SqCatch_Kg_lag1",
        "Industrial_Pressure",
        "Agricultural_Pressure"
    ],


    "full_pressures_plus_censoring": [
        "WaterTemp", "Depth", "SSH", "Chlor_a_mg_m3",
        "WaterTemp_lag1", "SSH_lag1", "Chlor_a_mg_m3_lag1",
        "SqCatch_Kg", "SqCatch_Kg_lag1",
         "Industrial_Pressure",
        "Agricultural_Pressure",
        "is_censored"
    ]
}



## Modeling Strategy

- Separate predictive models are trained for each pollutant to respect distinct chemical behaviors
- Models use time-aware cross-validation to avoid information leakage across sampling periods
- Regularized regression (ElasticNet) is applied to balance interpretability and stability under limited sample sizes
- Environmental, catch, and human pressure variables are evaluated in structured feature groups
- Seasonal structure is represented using continuous month-based encodings rather than categorical seasons
- Concentration values are log-transformed to reduce skew and stabilize variance
- Censored measurements (BLOD / BLOQ) are tracked explicitly and used for diagnostic and sensitivity analyses
- Model performance is evaluated using multiple metrics, with emphasis on robustness rather than peak accuracy



In [115]:
results = []

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

tscv = TimeSeriesSplit(n_splits=5)

for pollutant in final_df["pollutant"].unique():

    df_p = final_df[final_df["pollutant"] == pollutant].copy()
    y = df_p["concentration"]

    for model_name, features in FEATURE_SETS.items():

        X = df_p[features]

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(alpha=0.1, l1_ratio=0.5))
        ])

        r2s, maes, rmses = [], [], []

        for train_idx, test_idx in tscv.split(X):
            pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
            preds = pipeline.predict(X.iloc[test_idx])

            r2s.append(r2_score(y.iloc[test_idx], preds))
            maes.append(mean_absolute_error(y.iloc[test_idx], preds))
            rmses.append(np.sqrt(mean_squared_error(y.iloc[test_idx], preds)))

        results.append({
            "pollutant": pollutant,
            "model": model_name,
            "R2": np.mean(r2s),
            "MAE": np.mean(maes),
            "RMSE": np.mean(rmses)
        })

        pipeline.fit(X, y)
        joblib.dump(
            pipeline,
            model_dir / f"{pollutant}_{model_name}.joblib"
        )



In [116]:
#Model Comparison Table
results_df = pd.DataFrame(results)
results_df.sort_values(["pollutant", "model"])

,pollutant,model,R2,MAE,RMSE
1,Metal_A,env_plus_catch,-0.063125,2058.703641,2865.043665
0,Metal_A,environment_only,-0.063951,2054.992359,2866.342201
2,Metal_A,full_pressures,-0.059884,2046.964322,2859.578501
3,Metal_A,full_pressures_plus_censoring,-0.025900,2013.525707,2817.938940
5,Metal_B,env_plus_catch,0.004765,2.124183,2.656382
4,Metal_B,environment_only,0.004624,2.125356,2.656503
6,Metal_B,full_pressures,0.006401,2.111867,2.654309
7,Metal_B,full_pressures_plus_censoring,0.389187,1.391143,2.067158
9,Metal_C,env_plus_catch,-0.021933,458.053686,594.699501
8,Metal_C,environment_only,-0.021963,458.257865,594.717913


## Model Comparison Table

This table is used directly by the Streamlit app to:
- Rank models
- Support transparent decision-making
- Explain trade-offs to users


In [117]:
results_df = pd.DataFrame(results)
results_df.to_csv("../output/model_comparison_table.csv", index=False)

results_df.sort_values(["pollutant", "R2"], ascending=[True, False])


,pollutant,model,R2,MAE,RMSE
3,Metal_A,full_pressures_plus_censoring,-0.025900,2013.525707,2817.938940
2,Metal_A,full_pressures,-0.059884,2046.964322,2859.578501
1,Metal_A,env_plus_catch,-0.063125,2058.703641,2865.043665
0,Metal_A,environment_only,-0.063951,2054.992359,2866.342201
7,Metal_B,full_pressures_plus_censoring,0.389187,1.391143,2.067158
6,Metal_B,full_pressures,0.006401,2.111867,2.654309
5,Metal_B,env_plus_catch,0.004765,2.124183,2.656382
4,Metal_B,environment_only,0.004624,2.125356,2.656503
11,Metal_C,full_pressures_plus_censoring,0.059249,441.223069,569.880132
9,Metal_C,env_plus_catch,-0.021933,458.053686,594.699501


## Anomaly Detection Framework

Anomalies are identified using two complementary approaches:

1. **Model Residuals**
   Large deviations between observed and predicted concentrations.

2. **Laboratory Flags**
   Existing outlier labels provided by analytical laboratories.

This dual system distinguishes environmental anomalies from data-quality issues.

In [118]:
#Residual Based Anomaly Detection
anomaly_records = []

BASE_MODEL = "full_pressures"  # deliberately excludes is_censored

for pollutant in final_df["pollutant"].unique():

    df_p = final_df[final_df["pollutant"] == pollutant].copy()
    model_path = model_dir / f"{pollutant}_{BASE_MODEL}.joblib"

    if not model_path.exists():
        continue

    model = joblib.load(model_path)
    X = df_p[FEATURE_SETS[BASE_MODEL]]
    preds = model.predict(X)

    df_p["prediction"] = preds
    df_p["residual"] = df_p["concentration"] - preds

    threshold = 3 * df_p["residual"].std()

    df_p["model_anomaly"] = (
        (df_p["residual"].abs() > threshold) &
        (df_p["is_censored"] == 0)
    )

    anomaly_records.append(df_p)

In [100]:
# Combine anomaly signals
anomaly_df = pd.concat(anomaly_records)

anomaly_df["final_anomaly_flag"] = (
    anomaly_df["model_anomaly"] |
    (anomaly_df["outlier"].str.lower() == "yes")
)

anomaly_df['tissue'] = anomaly_df['ID'].str.rsplit('_', n=1).str[-1]

anomaly_df = anomaly_df[
    [
        "ID", "pollutant", "tissue",
        "concentration", "status", "outlier",
        "prediction", "residual",
        "model_anomaly", "final_anomaly_flag"
    ]
]


In [119]:
results_df.to_csv(
    "../output/model_comparison_table.csv",
    index=False
)

anomaly_df.to_csv(
    "../output/pollution_anomaly_table.csv",
    index=False
)
